# 04 - Market analysis: niches, comparable games and prices

Functions in `src/analysis.py`. Revenue = estimated net revenue, mid scenario.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.cleaning import load_clean
from src.estimation import add_estimates
from src.analysis import count_tags_loop, explode_tags, niche_table, tag_pair_table

# Number display: thousands separator and 2 decimals.
pd.options.display.float_format = '{:,.2f}'.format
df = add_estimates(load_clean('../data/games_clean.csv'))
len(df)

71352

# Part 1 - Niches

## 1. Count the tags by hand (loop + dictionary)

In [2]:
counts = count_tags_loop(df)
# The {tag: number of games} dictionary turned into a Series, sorted from most to least frequent.
top10_loop = pd.Series(counts).sort_values(ascending=False).head(10)
top10_loop

Singleplayer    44588
Indie           41885
Casual          32029
Action          31229
Adventure       30633
2D              24054
3D              16274
Simulation      15645
Strategy        15116
Puzzle          14593
dtype: int64

## 2. The same count with pandas

In [3]:
counts_pandas = explode_tags(df)['tag'].value_counts()
counts_pandas.head(10)

tag
Singleplayer    44588
Indie           41885
Casual          32029
Action          31229
Adventure       30633
2D              24054
3D              16274
Simulation      15645
Strategy        15116
Puzzle          14593
Name: count, dtype: int64

In [4]:
same = pd.Series(counts).sort_index().equals(counts_pandas.sort_index())
print('Identical counts:', same, '|', len(counts), 'different tags')

Identical counts: True | 451 different tags


## 3. Median or mean?

In [5]:
games_tags = explode_tags(df)
indie = games_tags[games_tags['tag'] == 'Indie']['revenue_mid']
print('Indie - mean revenue  :', round(indie.mean()), '$')
print('Indie - median revenue:', round(indie.median()), '$')

Indie - mean revenue  : 278376 $
Indie - median revenue: 1509 $


The mean is dozens of times higher than the median: a few multi-million hits crush it. The median describes the "typical" game, so that is the one we use.

## 4. The niche table (at least 50 games per tag, profitable = more than $100,000)

In [6]:
niches = niche_table(df)
print(len(niches), 'tags | minimum number of games:', niches['n_games'].min())
niches.head(15)

418 tags | minimum number of games: 50


,n_games,median_revenue,mean_revenue,profitable_share,recent_releases,previous_releases,growth
tag,,,,,,,
Games Workshop,62,"573,560.66","10,974,927.65",0.60,10,8,0.25
Kickstarter,133,"102,773.29","2,067,249.97",0.51,5,8,-0.38
Mod,50,"92,391.04","7,435,789.68",0.46,7,8,-0.12
Dwarf,76,"79,444.17","8,873,478.91",0.47,23,7,2.29
Cult Classic,173,"77,307.05","1,024,908.25",0.45,9,12,-0.25
Crowdfunded,81,"56,980.66","1,501,363.39",0.38,6,7,-0.14
Moddable,726,"49,310.38","5,649,913.68",0.43,212,164,0.29
Great Soundtrack,3167,"45,623.65","5,462,273.06",0.41,141,226,-0.38
Remake,332,"38,289.98","4,558,786.54",0.41,62,53,0.17


**Warning**: the first tags (Games Workshop, Kickstarter, Cult Classic, Great Soundtrack...) are not game concepts.
They are licence or funding labels, or labels given **after** the success (a game becomes a "Cult Classic" because it worked).
A studio cannot choose them: we set them aside when choosing niches.

In [7]:
# META_TAGS (defined in src/analysis.py): funding, licence or past-success tags, not genres.
from src.analysis import META_TAGS, promising_niches

meta_tags_present = []
for tag in META_TAGS:
    if tag in niches.index:
        meta_tags_present.append(tag)
genres = niches.drop(index=meta_tags_present)
print(len(meta_tags_present), 'tags removed |', len(genres), 'genres left')

17 tags removed | 401 genres left


## 5. Promising niches: few games, high median revenue, a good share of profitable games, and not filling up too fast

In [8]:
# promising_niches (src/analysis.py) removes the META_TAGS, then keeps the niches with
# at most 1,000 games, at least 30 % of profitable games, and releases that have not doubled in 2 years.
promising = promising_niches(niches)
promising.head(8)

,n_games,median_revenue,mean_revenue,profitable_share,recent_releases,previous_releases,growth
tag,,,,,,,
Villain Protagonist,204,"35,071.88","807,251.88",0.40,55,42,0.31
Real-Time with Pause,240,"33,072.35","4,788,802.77",0.40,46,45,0.02
Open World Survival Craft,497,"28,713.64","9,104,588.38",0.37,152,119,0.28
4X,349,"18,121.32","2,944,443.54",0.35,111,77,0.44
Heist,232,"14,065.38","3,515,057.79",0.34,69,64,0.08
Trains,285,"12,349.26","1,219,461.22",0.32,97,54,0.80
Motorbike,156,"12,227.64","1,043,492.77",0.35,42,43,-0.02
MMORPG,169,"12,022.10","7,924,067.61",0.31,57,34,0.68


**5 promising niches**:
1. **Real-Time with Pause** (240 games): median revenue ~$33,000, 40 % profitable games, stable releases: a loyal audience and few new competitors.
2. **Villain Protagonist** (204 games): ~$35,000 median, 40 % profitable: an original narrative angle that stands out.
3. **Open World Survival Craft** (497 games): ~$29,000 median, 37 % profitable: a widely played genre (Valheim), still far from saturation.
4. **4X** (349 games): ~$18,000 median, 35 % profitable: a strategy audience ready to pay, but releases are rising (+44 %).
5. **Heist** (232 games): ~$14,000 median, 34 % profitable, low growth (+8 %).

## 6. Niches to avoid: many games and very few profitable ones

In [9]:
to_avoid = genres[genres['n_games'] >= 1000].sort_values('profitable_share').head(8)
to_avoid

,n_games,median_revenue,mean_revenue,profitable_share,recent_releases,previous_releases,growth
tag,,,,,,,
Runner,1479,508.60,"21,591.79",0.03,529,501,0.06
Abstract,1977,904.93,"25,036.25",0.03,627,571,0.10
Clicker,2065,744.66,"22,379.69",0.03,796,669,0.19
Minimalist,5167,677.00,"48,592.75",0.03,1690,1683,0.00
Logic,4030,906.07,"68,211.13",0.03,1453,1330,0.09
Hidden Object,4081,"1,187.87","29,044.90",0.03,1524,1049,0.45
Linear,5683,792.67,"46,286.83",0.04,2444,1886,0.30
Precision Platformer,2108,755.24,"63,116.78",0.04,826,701,0.18


**3 niches to avoid**:
1. **Minimalist** (more than 5,000 games): median revenue ~$680, 3 % profitable games.
2. **Clicker** (more than 2,000 games): ~$740 median, 3 % profitable, and +19 % releases.
3. **Precision Platformer** (more than 2,000 games): ~$750 median, 4 % profitable.

To watch: **Roguelike Deckbuilder** releases have almost tripled (+170 %): the niche has been filling up very fast since the success of Balatro and Slay the Spire.

In [10]:
niches.loc[['Roguelike Deckbuilder', 'Rogue-like', 'Cozy', 'Deckbuilding']]

,n_games,median_revenue,mean_revenue,profitable_share,recent_releases,previous_releases,growth
tag,,,,,,,
Roguelike Deckbuilder,418,"14,938.37","556,264.91",0.22,208,77,1.70
Rogue-like,4565,"2,833.11","402,769.38",0.15,2094,910,1.30
Cozy,713,"4,610.65","176,629.91",0.18,378,114,2.32
Deckbuilding,1166,"4,599.88","369,319.26",0.17,591,244,1.42


## 7. Bonus: tag pairs

In [11]:
pairs = tag_pair_table(df)
pairs[pairs['n_games'] <= 1000].head(10)

,n_games,median_revenue,profitable_share
pair,,,
Multiplayer + Story Rich,542,"415,619.13",0.59
Multiplayer + Visual Novel,60,"128,791.21",0.52
Multiplayer + RPG,953,"94,110.66",0.50
Female Protagonist + Multiplayer,309,"94,027.88",0.50
FPS + Visual Novel,140,"80,158.87",0.47
Anime + Multiplayer,450,"71,705.65",0.46
Difficult + Multiplayer,528,"51,183.09",0.45
Anime + Open World,300,"49,733.55",0.46
Female Protagonist + Open World,438,"36,489.76",0.42


In [12]:
# Why do odd pairs (FPS + Visual Novel) earn so much?
games = df.copy()
games['n_tags'] = games['tags'].str.count(',') + 1   # number of commas + 1 = number of tags

# Spearman correlation compares rankings, so games with millions of reviews
# do not crush it (unlike the usual correlation).
correlation = games['n_tags'].corr(games['reviews'], method='spearman')
print('Correlation number of tags / number of reviews:', round(correlation, 2))

# Median revenue by number of tags (bands of 5 tags).
games['tag_band'] = pd.cut(games['n_tags'], bins=[0, 5, 10, 15, 20])
print(games.groupby('tag_band', observed=True)['revenue_mid'].median())

Correlation number of tags / number of reviews: 0.2
tag_band
(0, 5]     1,017.48
(5, 10]    1,341.90
(10, 15]     960.69
(15, 20]   2,469.85
Name: revenue_mid, dtype: float64


**Interpretation trap**: on Steam, players set the tags. The link between the number of tags and popularity is moderate (correlation 0.2), but games with more than 15 tags have a median revenue twice as high.
The unexpected pairs ("FPS + Visual Novel", "Multiplayer + Story Rich") therefore mostly appear on big games, which inflates their median revenue.
Conclusion: a pair is only useful if it describes a real concept (e.g. Roguelike + Deckbuilding); the raw ranking of pairs is not a list of niches to target.

In [13]:
wanted_pairs = []
for pair in ['Card Game + Rogue-like', 'Indie + Rogue-like', 'Cozy + Indie', 'Farming Sim + Indie']:
    if pair in pairs.index:
        wanted_pairs.append(pair)
pairs.loc[wanted_pairs]

,n_games,median_revenue,profitable_share
pair,,,
Indie + Rogue-like,2404,"3,016.44",0.16


# Part 2 - Comparable games and price analysis

## 1. Test `has_all_tags` on 4 hand-written cases

In [14]:
from src.analysis import find_comparables, has_all_tags, price_table, recommend_price, MIN_COMPARABLES

print(has_all_tags('Indie,Rogue-like,Action', ['Indie', 'Roguelike']))  # all the tags -> True
print(has_all_tags('Indie,Action', ['Indie', 'Roguelike']))             # a missing tag -> False
print(has_all_tags('Indie,Action', ['indie']))                          # different case -> True
print(has_all_tags('Indie,Action', []))                                 # empty list -> True

True
False
True
True


## 2. Our running example

A **2-person studio with a $60,000 budget** hesitates between two concepts sold at **$15**:
- **A**: a roguelike (tags Indie + Roguelike);
- **B**: a cozy game (tags Indie + Cozy).

In [15]:
CONCEPTS = {
    'A - Roguelike': ['Indie', 'Roguelike'],
    'B - Cozy': ['Indie', 'Cozy'],
    'Open World Survival Craft': ['Open World Survival Craft'],
    'Clicker': ['Clicker'],
}
for name, tags in CONCEPTS.items():
    all_prices = find_comparables(df, tags)
    near_price = find_comparables(df, tags, price=15)
    recent = find_comparables(df, tags, price=15, since_year=2020)
    print(name, '| tags only:', len(all_prices), '| price $15 ±50%:', len(near_price),
          '| and since 2020:', len(recent))
print('Reliable minimum of comparable games:', MIN_COMPARABLES)

A - Roguelike | tags only: 2404 | price $15 ±50%: 894 | and since 2020: 625


B - Cozy | tags only: 385 | price $15 ±50%: 125 | and since 2020: 119


Open World Survival Craft | tags only: 497 | price $15 ±50%: 284 | and since 2020: 216


Clicker | tags only: 2065 | price $15 ±50%: 272 | and since 2020: 234
Reliable minimum of comparable games: 30


There are far more than 30 comparable games for each concept. The minimum of 30 (constant `MIN_COMPARABLES`) is used in the simulator to warn the user when a niche is too small.

## 3. Revenue by price band

In [16]:
roguelike = find_comparables(df, ['Indie', 'Roguelike'])
print(len(roguelike), 'comparable games Indie + Roguelike')
price_table(roguelike)

2404 comparable games Indie + Roguelike


,n_games,median_revenue,profitable_share
price_band,,,
<5$,1196,754.11,0.02
5-10$,607,"5,475.52",0.10
10-15$,333,"38,684.52",0.35
15-20$,188,"209,891.02",0.60
20-30$,74,"984,828.73",0.82
30$+,6,"83,331.71",0.50


In [17]:
print('Recommended band:', recommend_price(roguelike))

Recommended band: 20-30$


In [18]:
for name, tags in CONCEPTS.items():
    print(name, '->', recommend_price(find_comparables(df, tags)))

A - Roguelike -> 20-30$


B - Cozy -> 15-20$


Open World Survival Craft -> 30$+


Clicker -> 30$+


In [19]:
price_table(find_comparables(df, ['Indie', 'Cozy']))

,n_games,median_revenue,profitable_share
price_band,,,
<5$,224,698.54,0.02
5-10$,73,"8,943.67",0.16
10-15$,40,"24,157.22",0.30
15-20$,38,"111,179.82",0.53
20-30$,9,"729,136.48",0.78
30$+,1,"582,448.48",1.00


**Reading**: in the Indie + Roguelike niche, a game at $15-20 earns a median of about **280 times more** than a game under $5 (≈ $210,000 against ≈ $750).
The recommended band depends on the niche: $20-30 for roguelikes, $15-20 for cozy games (the $20-30 band only has 9 games, below the minimum of 10).

**Caution**: this is a correlation, not a causal link. A game sold at $25 often has a bigger production budget and more content than a $3 game;
lowering your price will not make you sell more. The tool therefore recommends a band "where comparable games succeed", not a magic price.
For the same reason, very small bands (Clicker at $30 and more) give fragile recommendations.